In [1]:
!pip install torch transformers peft trl datasets bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.6 MB/s eta 0:00:00


In [2]:
!pip install --upgrade --force-reinstall --no-cache-dir torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto"
)

grpo_model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/CS272/GRPO-lora")
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/CS272/GRPO-lora")
grpo_model.eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [5]:
import re

def extract_answer(text):
    # extract final answer in #### format first
    match = re.search(r"####\s*([\d,.-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    # Fallback = last number in text in case model doesn't follow
    numbers = re.findall(r"\b\d+\.?\d*\b", text)
    return numbers[-1] if numbers else None

In [9]:
def output_answer(model, question):
  # prompts a model given a question and prints out model output
  messages = [{"role": "user", "content": f"{question}\nSolve step by step and end with #### <number>"}]
  prompt = tokenizer.apply_chat_template(
      messages,
      tokenize=False,
      add_generation_prompt=True
  )

  inputs = tokenizer(
      prompt,
      return_tensors="pt",
      truncation=True,
      max_length=512
  ).to(model.device)

  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=512,
          do_sample=False,
          pad_token_id=tokenizer.eos_token_id
      )

  generated = tokenizer.decode(
      outputs[0][inputs['input_ids'].shape[1]:],
      skip_special_tokens=True
  )

  pred = extract_answer(generated)

  print(f"Generated Output:\n{generated}")
  print()

In [12]:
question = "James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?"

output_answer(grpo_model, question)

Generated Output:
<think>James runs 3 x 60 = <<3*60=180>>180 meters per day.
He runs this 3 times a week so he runs 180 x 3 = <<180*3=540>>540 meters in one week.</think>#### 540

